# Всё вместе: BM25, эмбеддинги и реранкер

**Задача.** Для каждого из 2 452 тестовых запросов (`benchmark_queries.parquet`, ответов к ним у нас нет) нужно выбрать 50 объявлений из корпуса — 189 212 объявлений из `benchmark_items.parquet`. Качество меряем через Recall@50: какая доля правильных объявлений запроса (тех, что выбрал пользователь) попала в наши 50, в среднем по запросам. Recall@N — то же для первых N объявлений списка. Разницу двух таких долей пишем в процентных пунктах (п.п.).

**На чём проверяем.** Ответы известны только в train: 497 673 строки «поля поиска → выбранное объявление». Запрос — это набор полей поиска (текст, локация, фильтры, категория, флаг доставки). Строки с одинаковыми полями склеиваются в один запрос, и его правильные объявления — все выбранные в этих строках. Train поделён на три части (`validation/split.ipynb`):

- **валидация** — 5 032 проверочных запроса, похожих на тестовые, для замера качества;
- **обучение реранкера** — 6 487 запросов, на них учится LightGBM;
- **обучение ретривера** — остальные 482 161 строка: по ним считаются доли выборов по локациям и дообучается эмбеддер.

**Как устроено решение.** Сначала ретривер отбирает для запроса несколько сотен кандидатов — объявлений, из которых потом выбираются 50. Потом реранкер — модель LightGBM из `ranking/ranker.ipynb` — по набору чисел-признаков заново упорядочивает кандидатов, и первые 50 идут в ответ. В `ranking/ranker.ipynb` кандидатов давал один бейзлайн, а здесь их отбирают два поиска:

1. **Бейзлайн** из `retrieval/baseline.ipynb` — поиск по словам BM25. Оценка объявления тем выше, чем больше слов запроса в нём есть и чем эти слова реже. К ней прибавляются оценка по словам фильтров запроса (например, «Вид услуги Красота, здоровье») и поправка за локацию: насколько часто при поиске в этой локации пользователи выбирали объявления из локации кандидата (доли выборов по локациям). Берём 500 объявлений с наибольшей оценкой.
2. **Поиск по смыслу** дообученным эмбеддером из `embeddings/models.ipynb`. Эмбеддер — нейросеть, которая превращает текст в вектор. У похожих по смыслу текстов векторы близки, и косинус между ними ближе к 1. Оценка объявления — косинус между векторами запроса и объявления, делённый на лучший косинус по этому запросу (у лучшего объявления любого запроса получается 1), плюс та же поправка за локацию: `0.05 · log(доля выборов локации объявления + 0.0003)`. Вес 0.05 подобран в `embeddings/models.ipynb`. Добавка 0.0003 нужна для локаций, где ни разу не выбирали: их объявления сильно опускаются, но логарифм не уходит в минус бесконечность. Берём 200 объявлений с наибольшей оценкой.

Кандидаты запроса — объединение этих двух списков. Для каждой пары «запрос — кандидат» считаются все признаки из `ranking/ranker.ipynb` (их описание — в таблице раздела 2 там) и пять новых признаков эмбеддера:

| Признак | Что это |
|---|---|
| `dense_score` | косинус между вектором запроса и вектором объявления |
| `dense_relative` | косинус, делённый на максимальный косинус по запросу |
| `dense_combined` | оценка поиска по смыслу с поправкой за локацию, по ней выбираются первые 200 |
| `dense_rank` | место кандидата в списке поиска по смыслу, считая с 0 (200, если он не попал в первые 200) |
| `dense_microcat_share` | какая доля первых 30 кандидатов поиска по смыслу относится к той же подкатегории, что и этот кандидат |

**Эмбеддер почти не видел ответов.** Он дообучался только на обучении ретривера и почти не видел правильных ответов на проверочные запросы и запросы обучения реранкера: среди их пар «текст запроса — правильное объявление» в его обучающих данных нашлось только 0.55% и 1.2% (считались точные совпадения текста запроса вместе с фильтрами, `embeddings/models.ipynb`). Это важно: если бы эмбеддер знал ответы запросов обучения реранкера, его оценки на них были бы завышены, и реранкер научился бы слишком им доверять. Кандидатов и признаки считает `scripts/ranking/candidates.py`.

In [1]:
import polars as pl

from avito import paths
from avito.embeddings.load import DENSE_MODEL
from avito.ingestion import load as ingested
from avito.metrics.recall import bootstrap_interval, query_recalls
from avito.ranking.features import DENSE_FEATURES, RANKING_FEATURES
from avito.ranking.model import fit_ranker, rerank
from avito.validation.load import load_val_queries

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_tbl_width_chars(220);

Загружаем кандидатов с признаками для запросов обучения реранкера и для проверочных запросов. Одна строка — одна пара «запрос — кандидат», колонка `label` равна 1, если кандидат — правильное объявление:

In [2]:
ranker_candidates = pl.read_parquet(paths.candidates_path("ranker", DENSE_MODEL))
val_candidates = pl.read_parquet(paths.candidates_path("val", DENSE_MODEL))
val = load_val_queries()

pl.DataFrame({
    "набор": ["обучение реранкера", "валидация"],
    "запросов": [ranker_candidates["query_id"].n_unique(), val_candidates["query_id"].n_unique()],
    "пар «запрос — кандидат»": [ranker_candidates.height, val_candidates.height],
})

набор,запросов,пар «запрос — кандидат»
str,i64,i64
"""обучение реранкера""",6487,3798606
"""валидация""",5032,2998783


## 1. Сколько кандидатов и сколько среди них правильных

Списки бейзлайна (500 объявлений) и поиска по смыслу (200) частично совпадают, поэтому кандидатов на запрос меньше 700:

In [3]:
pool_sizes = val_candidates.group_by("query_id").len()["len"]

pl.DataFrame({
    "кандидатов на проверочный запрос": ["в среднем", "меньше всего", "больше всего"],
    "значение": [pool_sizes.mean(), pool_sizes.min(), pool_sizes.max()],
})

кандидатов на проверочный запрос,значение
str,f64
"""в среднем""",595.942568
"""меньше всего""",500.0
"""больше всего""",695.0


В среднем поиск по смыслу добавляет к 500 кандидатам бейзлайна ещё 96 новых, а остальные 104 из его 200 бейзлайн уже нашёл. Бывают запросы, где совпали все 200 (всего 500 кандидатов), и такие, где совпали только 5 (695 кандидатов).

Теперь смотрим, какая доля правильных объявлений попадает в каждый список: в первые 50 и во все кандидаты каждого поиска и в их объединение. Это Recall@N, где N — длина списка. Длина объединения у каждого запроса своя, в таблице для него — средняя.

In [4]:
def top_by(candidates: pl.DataFrame, column: str, depth: int) -> pl.DataFrame:
    top = candidates.filter(pl.col(column) < depth)
    return top.group_by("query_id", maintain_order=True).agg("item_id")


pool = val_candidates.group_by("query_id", maintain_order=True).agg("item_id")

pl.DataFrame([
    {"список": "бейзлайн (BM25 + локация + фильтры), первые 50", "объявлений в списке": 50, "доля правильных объявлений, попавших в список": query_recalls(top_by(val_candidates, "retrieval_rank", 50), val, k=50).mean()},
    {"список": "поиск по смыслу, первые 50", "объявлений в списке": 50, "доля правильных объявлений, попавших в список": query_recalls(top_by(val_candidates, "dense_rank", 50), val, k=50).mean()},
    {"список": "бейзлайн (BM25 + локация + фильтры), первые 500", "объявлений в списке": 500, "доля правильных объявлений, попавших в список": query_recalls(top_by(val_candidates, "retrieval_rank", 500), val, k=500).mean()},
    {"список": "поиск по смыслу, первые 200", "объявлений в списке": 200, "доля правильных объявлений, попавших в список": query_recalls(top_by(val_candidates, "dense_rank", 200), val, k=200).mean()},
    {"список": "все кандидаты (объединение двух списков)", "объявлений в списке": round(pool_sizes.mean()), "доля правильных объявлений, попавших в список": query_recalls(pool, val, k=10_000).mean()},
])

список,объявлений в списке,"доля правильных объявлений, попавших в список"
str,i64,f64
"""бейзлайн (BM25 + локация + фильтры), первые 50""",50,0.885135
"""поиск по смыслу, первые 50""",50,0.894376
"""бейзлайн (BM25 + локация + фильтры), первые 500""",500,0.9687
"""поиск по смыслу, первые 200""",200,0.964361
"""все кандидаты (объединение двух списков)""",596,0.987977


Поиск по смыслу сам по себе ставит в первые 50 больше правильных объявлений, чем бейзлайн: 0.894 против 0.885. Объединение двух списков содержит 98.8% правильных объявлений, а первые 500 бейзлайна — 96.9%. Объединение длиннее, но дело не только в длине: даже в первых 1000 объявлениях бейзлайна 98.0% правильных (`retrieval/baseline.ipynb`). Реранкер только переставляет кандидатов, поэтому 98.8% — предел, выше которого его Recall@50 не поднимется.

## 2. Обучение и оценка

Функции те же, что в `ranking/ranker.ipynb`. Для каждого варианта запоминаем Recall@50 на валидации и 95% интервал для него. Интервал показывает, насколько Recall@50 мог бы сдвинуться на других проверочных запросах: 1000 раз случайно перевыбираем 5 032 запроса с повторениями и каждый раз пересчитываем Recall@50 (бутстреп, `retrieval/baseline.ipynb`). Первые две строки таблицы — первые 50 кандидатов каждого поиска, без реранкера.

In [5]:
results = []
recalls_by_method = {}
predictions_by_method = {}
models = {}
baseline = "бейзлайн (BM25 + локация + фильтры), первые 50"


def evaluate(name: str, predictions: pl.DataFrame) -> pl.DataFrame:
    recalls = query_recalls(predictions, val)
    low, high = bootstrap_interval(recalls)
    recalls_by_method[name] = recalls
    predictions_by_method[name] = predictions
    results.append({"вариант": name, "Recall@50": recalls.mean(), "95% интервал для Recall@50": f"[{low:.3f}, {high:.3f}]"})
    return pl.DataFrame(results)


evaluate(baseline, top_by(val_candidates, "retrieval_rank", 50))
evaluate("поиск по смыслу, первые 50", top_by(val_candidates, "dense_rank", 50))

вариант,Recall@50,95% интервал для Recall@50
str,f64,str
"""бейзлайн (BM25 + локация + фильтры), первые 50""",0.885135,"""[0.876, 0.894]"""
"""поиск по смыслу, первые 50""",0.894376,"""[0.886, 0.903]"""


Реранкер учится на запросах обучения реранкера, как в `ranking/ranker.ipynb`. 10% из них (те, у которых номер делится на 10) в обучение не идут. LightGBM добавляет деревья решений одно за другим и после каждого считает качество на этих отложенных запросах. Если 100 деревьев подряд оно не растёт, обучение останавливается, и модель оставляет деревья до лучшего места. Иначе модель начала бы запоминать обучающие запросы вместо общих закономерностей.

Вторая функция нужна для раздела 5: там в обучение добавляется половина проверочных запросов.

In [6]:
early_stopping_mask = pl.col("query_id").str.slice(1).cast(pl.Int64) % 10 == 0
ranker_train = ranker_candidates.filter(~early_stopping_mask)
ranker_early_stopping = ranker_candidates.filter(early_stopping_mask)
val_half = pl.col("query_id").cast(pl.Int64) % 2
ALL_FEATURES = RANKING_FEATURES + DENSE_FEATURES


def train_and_rerank(name: str, features: list[str]) -> pl.DataFrame:
    model = fit_ranker(ranker_train, ranker_early_stopping, features=features)
    models[name] = model
    return rerank(val_candidates, model.booster_, features=features)


def train_with_half_of_val(name: str, features: list[str]) -> pl.DataFrame:
    predictions = []
    for half in [0, 1]:
        train = pl.concat([ranker_train, val_candidates.filter(val_half != half)])
        model = fit_ranker(train, ranker_early_stopping, features=features)
        predictions.append(rerank(val_candidates.filter(val_half == half), model.booster_, features=features))
    models[name] = model
    return pl.concat(predictions)

## 3. Больше кандидатов, признаки прежние

Сначала проверяем, что даёт само расширение списка кандидатов: реранкер с прежними признаками из `ranking/ranker.ipynb`, без признаков эмбеддера, но на кандидатах обоих поисков. В `ranking/ranker.ipynb` на кандидатах одного бейзлайна он давал 0.907.

In [7]:
evaluate("реранкер: прежние признаки (без эмбеддера), кандидаты обоих поисков", train_and_rerank("прежние признаки", RANKING_FEATURES))

вариант,Recall@50,95% интервал для Recall@50
str,f64,str
"""бейзлайн (BM25 + локация + фильтры), первые 50""",0.885135,"""[0.876, 0.894]"""
"""поиск по смыслу, первые 50""",0.894376,"""[0.886, 0.903]"""
"""реранкер: прежние признаки (без эмбеддера), кандидаты обоих поисков""",0.91246,"""[0.905, 0.920]"""


Одно расширение списка кандидатов даёт немного: 0.907 → 0.912. Значима ли эта разница, мы не проверяли: модель с 0.907 обучена в другом ноутбуке на других кандидатах. Вероятно, по прежним признакам трудно отличить хорошего кандидата из поиска по смыслу от плохого. Поэтому дальше добавляем признаки эмбеддера.

## 4. Добавляем признаки эмбеддера

In [8]:
evaluate("реранкер: + признаки эмбеддера", train_and_rerank("+ признаки эмбеддера", ALL_FEATURES))

вариант,Recall@50,95% интервал для Recall@50
str,f64,str
"""бейзлайн (BM25 + локация + фильтры), первые 50""",0.885135,"""[0.876, 0.894]"""
"""поиск по смыслу, первые 50""",0.894376,"""[0.886, 0.903]"""
"""реранкер: прежние признаки (без эмбеддера), кандидаты обоих поисков""",0.91246,"""[0.905, 0.920]"""
"""реранкер: + признаки эмбеддера""",0.945648,"""[0.939, 0.952]"""


Признаки эмбеддера дают +3.3 п.п.: 0.912 → 0.946. После учёта локации в бейзлайне (`retrieval/baseline.ipynb`) это самый большой шаг: фильтры и каждая группа признаков реранкера давали не больше 1.2 п.п. (`ranking/ranker.ipynb`). Значит, чтобы правильно расставить кандидатов, реранкеру нужна похожесть запроса и объявления по смыслу.

## 5. Добавляем в обучение половину валидации

Запросы обучения реранкера устроены не так, как тест. Почти все их тексты (97%) встречаются в train: при делении train все новые тексты забрала валидация (`validation/split.ipynb`). Эмбеддер видел при обучении 89% этих текстов, хотя в основном в паре с другими объявлениями, а не с правильными ответами этих запросов. Среди текстов валидации эмбеддеру знакомы только 32%, среди тестовых — 28% (`embeddings/models.ipynb`). Это меньше, чем 37% текстов валидации, которые встречаются в train (`validation/split.ipynb`; в разделе 8 это группа «текст встречается в train», 1 861 запрос из 5 032). Там текст запроса ищется во всём train без проверочных запросов, а здесь — только среди текстов, на которых учился эмбеддер, и вместе с фильтрами.

Сравниваем первые 50 объявлений поиска по смыслу на знакомых и незнакомых эмбеддеру текстах валидации. Для сравнения — первые 50 бейзлайна, который на текстах запросов не учился:

In [9]:
embedder_texts = pl.read_parquet(paths.EMBEDDING_PAIRS, columns=["query_text"])["query_text"].unique()
val_texts = pl.read_parquet(paths.EMBEDDING_QUERIES, columns=["query_id", "set", "text"]).filter(pl.col("set") == "val")
by_text = val.select("query_id").join(val_texts, on="query_id", how="left", maintain_order="left").with_columns(
    pl.col("text").is_in(embedder_texts.implode()).alias("embedder_saw_text"),
    pl.Series("baseline", recalls_by_method[baseline]),
    pl.Series("dense", recalls_by_method["поиск по смыслу, первые 50"]),
)

pl.DataFrame([
    {
        "проверочные запросы": name,
        "запросов": part.height,
        "Recall@50, бейзлайн": part["baseline"].mean(),
        "Recall@50, поиск по смыслу": part["dense"].mean(),
        "поиск по смыслу лучше бейзлайна на": part["dense"].mean() - part["baseline"].mean(),
    }
    for name, part in [
        ("текст эмбеддер видел при обучении", by_text.filter(pl.col("embedder_saw_text"))),
        ("текст эмбеддер не видел", by_text.filter(~pl.col("embedder_saw_text"))),
    ]
])

проверочные запросы,запросов,"Recall@50, бейзлайн","Recall@50, поиск по смыслу",поиск по смыслу лучше бейзлайна на
str,i64,f64,f64,f64
"""текст эмбеддер видел при обучении""",1614,0.899215,0.916667,0.017451
"""текст эмбеддер не видел""",3418,0.878486,0.88385,0.005364


На текстах, которые эмбеддер видел, поиск по смыслу лучше бейзлайна на 1.7 п.п. (0.917 против 0.899), а на незнакомых — только на 0.5 п.п. (0.884 против 0.878). Значит, на знакомых текстах эмбеддер, вероятно, точнее; значимость этой разницы мы не проверяли. Тогда реранкер, обученный почти только на знакомых текстах, может придавать признакам эмбеддера слишком большой вес и чаще ошибаться на новых текстах, а в тесте их большинство.

Проверяем, помогает ли добавить в обучение запросы, похожие на тест. Делим проверочные запросы на две половины (по чётности номера), добавляем одну половину в обучение, а качество считаем на другой. Потом меняем половины местами. Так каждый проверочный запрос оценивает модель, которая его не видела.

In [10]:
evaluate("реранкер: + признаки эмбеддера, + половина валидации в обучении", train_with_half_of_val("+ половина валидации", ALL_FEATURES))

вариант,Recall@50,95% интервал для Recall@50
str,f64,str
"""бейзлайн (BM25 + локация + фильтры), первые 50""",0.885135,"""[0.876, 0.894]"""
"""поиск по смыслу, первые 50""",0.894376,"""[0.886, 0.903]"""
"""реранкер: прежние признаки (без эмбеддера), кандидаты обоих поисков""",0.91246,"""[0.905, 0.920]"""
"""реранкер: + признаки эмбеддера""",0.945648,"""[0.939, 0.952]"""
"""реранкер: + признаки эмбеддера, + половина валидации в обучении""",0.95045,"""[0.944, 0.956]"""


С половиной валидации в обучении Recall@50 = 0.950, на 0.5 п.п. больше, чем без неё (0.946). Это согласуется с предположением выше, но у роста может быть и другая причина: обучающих запросов стало больше — к 90% из 6 487 запросов обучения реранкера добавилась половина из 5 032 проверочных. Этот опыт две причины не разделяет.

В любом случае учить реранкер и на валидации полезно, поэтому итоговая модель для ответа (`scripts/ranking/train.py`) учится на запросах обучения реранкера и всей валидации. Её Recall@50 на валидации был бы завышен: она видела ответы всех проверочных запросов. Самая точная оценка её качества, которая у нас есть, — 0.950 из этого раздела.

## 6. Не случайны ли улучшения

Для каждого проверочного запроса считаем, насколько изменился его Recall@50 при переходе от одного варианта к другому, и тем же бутстрепом строим 95% интервал для средней разницы (как в `retrieval/baseline.ipynb`). Если интервал целиком выше 0, улучшение, скорее всего, не случайное.

In [11]:
def compare(before: str, after: str) -> dict:
    difference = recalls_by_method[after] - recalls_by_method[before]
    low, high = bootstrap_interval(difference)
    return {"было": before, "стало": after, "разница Recall@50": difference.mean(), "95% интервал разницы": f"[{low:.4f}, {high:.4f}]"}


pl.DataFrame([
    compare(baseline, "поиск по смыслу, первые 50"),
    compare(baseline, "реранкер: прежние признаки (без эмбеддера), кандидаты обоих поисков"),
    compare("реранкер: прежние признаки (без эмбеддера), кандидаты обоих поисков", "реранкер: + признаки эмбеддера"),
    compare("реранкер: + признаки эмбеддера", "реранкер: + признаки эмбеддера, + половина валидации в обучении"),
])

было,стало,разница Recall@50,95% интервал разницы
str,str,f64,str
"""бейзлайн (BM25 + локация + фильтры), первые 50""","""поиск по смыслу, первые 50""",0.009241,"""[0.0001, 0.0189]"""
"""бейзлайн (BM25 + локация + фильтры), первые 50""","""реранкер: прежние признаки (без эмбеддера), кандидаты обоих поисков""",0.027325,"""[0.0212, 0.0340]"""
"""реранкер: прежние признаки (без эмбеддера), кандидаты обоих поисков""","""реранкер: + признаки эмбеддера""",0.033188,"""[0.0266, 0.0396]"""
"""реранкер: + признаки эмбеддера""","""реранкер: + признаки эмбеддера, + половина валидации в обучении""",0.004803,"""[0.0019, 0.0078]"""


Все интервалы целиком выше 0, то есть каждый шаг, скорее всего, действительно улучшает результат. Слабее всего разница между самими поисками без реранкера: интервал [0.0001, 0.0189] почти касается 0, так что по первым 50 объявлениям они примерно равны. Зато реранкер на кандидатах обоих поисков и с признаками эмбеддера даёт 0.946 — больше, чем любой из поисков по отдельности (0.885 и 0.894).

## 7. Какие признаки важнее всего

Смотрим на модель из раздела 5, обученную на запросах обучения реранкера и одной из половин валидации (вторую из двух). Модель состоит из деревьев решений, и каждое дерево несколько раз делит кандидатов по значению какого-нибудь признака. Вклад признака — насколько в сумме такие деления по нему улучшили порядок кандидатов. В таблице — доля вклада признака от суммарного вклада всех признаков. Прежние признаки описаны в таблице раздела 2 `ranking/ranker.ipynb`, новые — в начале этого ноутбука.

In [12]:
gain = models["+ половина валидации"].booster_.feature_importance("gain")
pl.DataFrame({"признак": ALL_FEATURES, "доля вклада": gain / gain.sum()}).sort("доля вклада", descending=True)

признак,доля вклада
str,f64
"""dense_rank""",0.403563
"""retrieval_rank""",0.31777
"""dense_combined""",0.069212
"""distance_km""",0.028699
"""retrieval_score""",0.02538
"""dense_relative""",0.018764
"""log_prior""",0.014956
"""same_location_share""",0.013337
"""coverage_full""",0.012546


Главный признак теперь — место кандидата в списке поиска по смыслу (`dense_rank`, 40% вклада), второй — место в выдаче бейзлайна (`retrieval_rank`, 32%). Модель опирается на оба поиска сразу. Дальше идут оценка поиска по смыслу с поправкой за локацию (`dense_combined`, 7%) и расстояние до центра локации поиска (`distance_km`, 3%). Центр — медиана координат объявлений, которые выбирали при поиске в этой локации. `search_category` модель не использует совсем: почти все обучающие запросы ищут в «Услугах» (`search_category` = 114).

## 8. Где ещё теряем

Для лучшего варианта (реранкер с признаками эмбеддера и половиной валидации в обучении) смотрим, что случилось с каждым правильным объявлением валидации: попало в ответ (первые 50), было среди кандидатов, но реранкер поставил его ниже 50-го места, или не попало в кандидаты вообще. Доли здесь считаются по всем правильным объявлениям вместе, а не по запросам. Поэтому первая строка чуть отличается от Recall@50 (0.950), который сначала считается для каждого запроса, а потом усредняется.

In [13]:
best = "реранкер: + признаки эмбеддера, + половина валидации в обучении"
in_top = predictions_by_method[best].explode("item_id", empty_as_null=True).with_columns(pl.lit(True).alias("in_top"))
in_pool = val_candidates.select("query_id", "item_id").with_columns(pl.lit(True).alias("in_pool"))

relevant = val.select("query_id", pl.col("relevant_item_ids").alias("item_id")).explode("item_id", empty_as_null=True)
relevant = relevant.join(in_top, on=["query_id", "item_id"], how="left").join(in_pool, on=["query_id", "item_id"], how="left")
relevant = relevant.with_columns(pl.col("in_top").fill_null(False), pl.col("in_pool").fill_null(False))

pl.DataFrame({
    "что случилось с правильным объявлением": ["попало в первые 50", "было среди кандидатов, но ниже 50-го места", "не попало в кандидаты"],
    "доля правильных объявлений": [
        relevant["in_top"].mean(),
        (relevant["in_pool"] & ~relevant["in_top"]).mean(),
        (~relevant["in_pool"]).mean(),
    ],
})

что случилось с правильным объявлением,доля правильных объявлений
str,f64
"""попало в первые 50""",0.950707
"""было среди кандидатов, но ниже 50-го места""",0.037313
"""не попало в кандидаты""",0.01198


95.1% правильных объявлений попадают в ответ. 3.7% есть среди кандидатов, но реранкер ставит их ниже 50-го места: это можно выиграть только лучшим реранкером. 1.2% не находит ни один из двух поисков: это можно выиграть только лучшим поиском. Потери реранкера примерно втрое больше, поэтому дальше выгоднее улучшать реранкер.

Теперь Recall@50 по группам проверочных запросов, как в `ranking/ranker.ipynb`: встречается ли текст запроса в train (без проверочных запросов); задал ли пользователь фильтры (например, «Вид услуги»); лежат ли все правильные объявления запроса в локации поиска или хотя бы одно — в другой; сколько слов в запросе. Это группы из `retrieval/baseline.ipynb`, только без двух групп про поиск по региону или стране, а запросы из 2 и 3 слов объединены в одну группу. Сравниваем первые 50 бейзлайна и лучший вариант — реранкер с признаками эмбеддера и половиной валидации в обучении.

In [14]:
item_locations = ingested.load_items(["item_id", "item_location_id"])
relevant_locations = val.select("query_id", "search_location_id", pl.col("relevant_item_ids").alias("item_id")).explode("item_id", empty_as_null=True)
relevant_locations = relevant_locations.join(item_locations, on="item_id")
relevant_locations = relevant_locations.group_by("query_id").agg(
    (pl.col("item_location_id") == pl.col("search_location_id")).all().alias("relevant_in_search_location")
)

strata = val.select("query_id", "seen_text", "search_infm_params_text", "query_stems")
strata = strata.join(relevant_locations, on="query_id", how="left", maintain_order="left")
strata = strata.with_columns(
    pl.Series("baseline", recalls_by_method[baseline]),
    pl.Series("final", recalls_by_method[best]),
    pl.col("query_stems").str.split(" ").list.len().alias("words"),
)

groups = {
    "все запросы": pl.lit(True),
    "текст встречается в train": pl.col("seen_text"),
    "текст новый": ~pl.col("seen_text"),
    "без фильтров": pl.col("search_infm_params_text") == "",
    "с фильтрами": pl.col("search_infm_params_text") != "",
    "все правильные объявления в локации поиска": pl.col("relevant_in_search_location"),
    "хотя бы одно правильное объявление в другой локации": ~pl.col("relevant_in_search_location"),
    "запрос из 1 слова": pl.col("words") == 1,
    "запрос из 2–3 слов": pl.col("words").is_between(2, 3),
    "запрос из 4 слов и больше": pl.col("words") >= 4,
}

pl.DataFrame([
    {
        "группа запросов": name,
        "запросов": strata.filter(condition).height,
        "Recall@50, бейзлайн": strata.filter(condition)["baseline"].mean(),
        "Recall@50, реранкер с половиной валидации": strata.filter(condition)["final"].mean(),
    }
    for name, condition in groups.items()
])

группа запросов,запросов,"Recall@50, бейзлайн","Recall@50, реранкер с половиной валидации"
str,i64,f64,f64
"""все запросы""",5032,0.885135,0.95045
"""текст встречается в train""",1861,0.900233,0.95549
"""текст новый""",3171,0.876275,0.947493
"""без фильтров""",2634,0.882055,0.950519
"""с фильтрами""",2398,0.888518,0.950375
"""все правильные объявления в локации поиска""",3899,0.926349,0.977045
"""хотя бы одно правильное объявление в другой локации""",1133,0.743307,0.858929
"""запрос из 1 слова""",192,0.776042,0.90625
"""запрос из 2–3 слов""",3985,0.8867,0.952238


Лучший вариант сильнее всего помог там, где бейзлайн был слабее всего: на запросах из одного слова (0.78 → 0.91) и на запросах, где хотя бы одно правильное объявление лежит не в локации поиска (0.74 → 0.86). В этот прирост входит и сам реранкер: без поиска по смыслу (`ranking/ranker.ipynb`) он поднимал эти группы только до 0.80 и 0.80. Остальное дали поиск по смыслу, признаки эмбеддера и обучение на половине валидации.

Разрыв между новыми и знакомыми текстами сократился с 2.4 п.п. (0.876 против 0.900 у бейзлайна) до 0.8 п.п. (0.947 против 0.955). Это важно, потому что в тесте 63% текстов новые (`validation/split.ipynb`).

## Итог

Все варианты вместе:

In [15]:
pl.DataFrame(results)

вариант,Recall@50,95% интервал для Recall@50
str,f64,str
"""бейзлайн (BM25 + локация + фильтры), первые 50""",0.885135,"""[0.876, 0.894]"""
"""поиск по смыслу, первые 50""",0.894376,"""[0.886, 0.903]"""
"""реранкер: прежние признаки (без эмбеддера), кандидаты обоих поисков""",0.91246,"""[0.905, 0.920]"""
"""реранкер: + признаки эмбеддера""",0.945648,"""[0.939, 0.952]"""
"""реранкер: + признаки эмбеддера, + половина валидации в обучении""",0.95045,"""[0.944, 0.956]"""


- Кандидаты — объединение первых 500 объявлений бейзлайна и первых 200 поиска по смыслу, в среднем 596 на запрос. Среди них 98.8% правильных объявлений, среди первых 500 бейзлайна — 96.9%.
- Одно расширение списка кандидатов даёт немного (0.907 → 0.912), главное дают признаки эмбеддера: +3.3 п.п., до 0.946. Самый важный признак — место кандидата в списке поиска по смыслу.
- Если учить реранкер ещё и на половине валидации, Recall@50 растёт на 0.5 п.п., до 0.950 (95% интервал [0.944, 0.956]). Вероятно, помогает то, что тексты валидации, как и тестовые, эмбеддеру в основном незнакомы: он видел 89% текстов запросов обучения реранкера, но только 32% текстов валидации и 28% тестовых. Часть прироста может давать и то, что обучающих запросов стало больше. Итоговая модель для ответа учится на запросах обучения реранкера и всей валидации.
- На лидерборде — оценке организаторов по скрытым ответам тестовых запросов — итоговая модель дала 0.922, на 2.8 п.п. ниже, чем 0.950 на валидации. Вероятно, отчасти это из-за того, чем валидация не похожа на тест (`validation/split.ipynb`): в тесте 9% запросов ищут по всему Авито, а в валидации таких почти нет, и запросов с несколькими правильными объявлениями в тесте, судя по условию задачи, больше. Отдельно мы это не проверяли.
- Осталось: 3.7% правильных объявлений есть среди кандидатов, но не в первых 50, и 1.2% не находит ни один поиск.